In [ ]:
1

In [ ]:
import json
import shutil
from pathlib import Path

import anndata as ad
import numpy as np
import os
import pandas as pd

In [ ]:
from tqdm import tqdm

In [ ]:
def filter_samples(adata):
    adata = adata.copy()
    perturbagen_mask = ~adata.obs['pubchem_cid'].isna()
    dose_mask = ~(adata.obs['pert_dose_uM'] == 0)
    return adata[perturbagen_mask & dose_mask].copy()

In [ ]:
def construct_df(adata):
    adata = adata.copy()

    dataset = adata.obs['dataset'].to_numpy()
    context = adata.obs['cell_type'].to_numpy()
    perturbations = adata.obs['pubchem_cid'].to_numpy()
    dose = adata.obs['pert_dose_uM'].to_numpy()
    time = adata.obs['pert_time_h'].to_numpy()

    values = adata.layers[LAYER]
    readout_names = adata.var.index.to_numpy()

    n_perts, n_readouts = values.shape

    df = pd.DataFrame({
    'dataset':  np.repeat(dataset, n_readouts),
    'context': np.repeat(context, n_readouts),
    'perturbation': np.repeat(perturbations, n_readouts),
    'log_dose': np.log10(np.repeat(dose, n_readouts)),
    'time': np.repeat(time, n_readouts),
    'readout': np.tile(readout_names, n_perts),
    'value': values.ravel(),
    })
    return df

In [ ]:
def filter_readout(df):
    df = df.copy()

    na_mask = df.isna().any(axis=1)
    #duplicates_mask = df[['dataset', 'context', 'perturbation', 'dose', 'time', 'readout']].duplicated()
    readouts_mask = na_mask
    
    if df[readouts_mask].shape[0] != 0:
        print(f'Missing values: {df[readouts_mask].shape[0]}')
        
    #readouts_mask = readouts_mask | duplicates_mask
    #if df[readouts_mask].shape[0] != 0:
    #    print(f'Duplicates: {df[readouts_mask].shape[0]}')
    
    df = df[~readouts_mask].copy()

    return df

In [ ]:
def run_construct_df(adata):
    
    adata = filter_samples(adata)
    df = construct_df(adata)
    df = filter_readout(df)

    return df

In [ ]:
LAYER = 'logFC'
PLIBDATA_ROOT = '../.plib_cache/raw_datasets'
BASE_PATH = '/lustre/groups/ml01/workspace/olga.novitskaia/data_updated'

#deg_folders = os.listdir(BASE_PATH)

In [ ]:
dict_paths = {
#'tahoe': f'{BASE_PATH}/tahoe/deg_data/group_rep/full/qc_false/filter_min_cells_50/results/',
'l1000_phase1': f'{BASE_PATH}/l1000_phase1/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'l1000_phase2': f'{BASE_PATH}/l1000_phase2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
#'novartis': f'{BASE_PATH}/novartis_batch_2500/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'vcpi_0001': f'{BASE_PATH}/vcpi_0001/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'vcpi_0002': f'{BASE_PATH}/vcpi_0002/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'gdpx2': f'{BASE_PATH}/gdpx2/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'cigs_mce': f'{BASE_PATH}/cigs_mce/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',
'cigs_tcm': f'{BASE_PATH}/cigs_tcm/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'dili_train': f'{BASE_PATH}/dilimap_train/deg_data/group_rep/full/qc_false/filter_min_cells_0/results/',

'sciplex': f'{BASE_PATH}/sciplex/deg_data/group_rep/full/qc_false/filter_min_cells_10/results/',
}

In [ ]:
os.makedirs(PLIBDATA_ROOT, exist_ok=True)

In [ ]:
for i, key in enumerate(dict_paths.keys()):
    print(f'{i} out of {len(dict_paths.keys())}')
    for j, file in enumerate(os.listdir(dict_paths[key])):
        print(f'{j} out of {len(os.listdir(dict_paths[key]))}')
        path = dict_paths[key] + file
        adata = ad.read_h5ad(path)
        df = run_construct_df(adata)
        
        os.makedirs(f'{PLIBDATA_ROOT}/{key}/', exist_ok=True)
        df.to_parquet(f'{PLIBDATA_ROOT}/{key}/{file.split('_de')[0]}.parquet')
        #break